<a href="https://colab.research.google.com/github/adriansivina-droid/scraper-mef/blob/main/FISCAL/PIM_Funcion_LOR_SMN_UCA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ═══ BLOQUE 1 · INSTALACIÓN (ejecutar una vez por sesión) ═══
!pip install requests beautifulsoup4 pandas numpy tqdm google-api-python-client --quiet
print("✅ Librerías listas")
import os, re, time, random, zipfile, unicodedata
import requests
import numpy as np
import pandas as pd
from datetime import date
from bs4 import BeautifulSoup
from tqdm.auto import tqdm
from IPython.display import display

In [ ]:
# ═══ BLOQUE 2 · CONFIGURACIÓN, EXTRACCIÓN Y CIERRE ═══
# ── 1. CONFIGURACIÓN ─────────────────────────────────────────────────────────
# El PIM es un monto ANUAL: se consulta sin filtro de mes.
AÑO_INICIO = 2026
AÑO_FIN    = 2026

CONTROL = True                 # suma de niveles vs total del departamento (todos los niveles)
SUBIR_A_DRIVE = True
DRIVE_FOLDER_ID = "1Vu6zpHljX_178x0ZDTbtE9BL2b0pS0uh"           # CSV de datos y LOG
DRIVE_FOLDER_CONTROL_ID = "1lHTsdrjZgCmJDOh_kbHhhG1-eJDQg2jz"   # archivos CONTROL
DESCARGA_FINAL = "ninguna"     # "zip" | "individual" | "ninguna"

PAUSA = (0.4, 1.0)             # segundos aleatorios entre consultas (cortesía con el servidor)

# ── Cómo arma la URL Consulta Amigable ───────────────────────────────────────
# 24=Genérica, 21=Departamento (Meta), 1=Nivel de gobierno, 8=Función.
# La PRIMERA clave con valor vacío es la columna que se lista (aquí, 8 = Función).
GENERICA = "6-2-6"                            # Genérica 6-26
CLAVE_DEPTO = "21"                            # Departamento (Meta), como el código original
CLAVE_FUNCION = "8"
PARAMS_EXTRA = {}   # si F12>Network muestra otros parámetros, se sobreescriben aquí

# ── Catálogos (nombres como en el código original) ───────────────────────────
departamentos = [
    {"codigo": "16", "departamento": "16: LORETO"},
    {"codigo": "22", "departamento": "22: SAN MARTIN"},
    {"codigo": "25", "departamento": "25: UCAYALI"},
]
niveles = [
    {"codigo": "E", "nivel": "GOBIERNO NACIONAL"},
    {"codigo": "M", "nivel": "GOBIERNOS LOCALES"},
    {"codigo": "R", "nivel": "GOBIERNOS REGIONALES"},
]
SUFIJO_REGIONES = "LOR - SMN - UCA"

hoy = date.today()
if not (2007 <= AÑO_INICIO <= AÑO_FIN <= hoy.year):
    raise ValueError(f"Rango de años inválido ({AÑO_INICIO}-{AÑO_FIN}). Máximo: {hoy.year}.")
años = list(range(AÑO_INICIO, AÑO_FIN + 1))

ruta_guardado = "/content/PIM_Funcion_LOR_SMN_UCA"
os.makedirs(ruta_guardado, exist_ok=True)

def nombre_archivo_salida(año):
    return f"PIM Funcion {año} - {SUFIJO_REGIONES}.csv"

COLUMNAS_SALIDA = ["AÑO", "DEPARTAMENTO", "NIVEL DE GOBIERNO", "FUNCION", "PIM"]
print(f"Años: {', '.join(map(str, años))}  |  regiones: {SUFIJO_REGIONES}  |  "
      f"{len(años) * len(departamentos) * len(niveles)} consultas de datos")

# ── 2. GOOGLE DRIVE ──────────────────────────────────────────────────────────
drive_subidos, drive_fallidos = [], []
STATUS_REINTENTABLES = {403, 429, 500, 502, 503, 504}

def conectar_drive(folder_ids):
    """Autentica y verifica permiso de Editor en cada carpeta."""
    from google.colab import auth
    from googleapiclient.discovery import build
    from googleapiclient.errors import HttpError
    auth.authenticate_user()
    servicio = build("drive", "v3", cache_discovery=False)
    for folder_id in dict.fromkeys(folder_ids):
        try:
            carpeta = servicio.files().get(
                fileId=folder_id, fields="id,name,mimeType,capabilities(canAddChildren)",
                supportsAllDrives=True).execute()
        except HttpError as e:
            raise PermissionError(f"Sin acceso a la carpeta de Drive {folder_id} (HTTP {e.resp.status}).") from e
        if carpeta.get("mimeType") != "application/vnd.google-apps.folder":
            raise ValueError(f"El ID {folder_id} no corresponde a una carpeta de Drive.")
        if not carpeta.get("capabilities", {}).get("canAddChildren", False):
            raise PermissionError(f"Sin permiso de Editor en '{carpeta['name']}'.")
        print(f"☁️  Drive conectado -> carpeta: '{carpeta['name']}'")
    print()
    return servicio

def subir_a_drive(servicio, ruta_local, folder_id, max_reintentos=5):
    from googleapiclient.errors import HttpError
    from googleapiclient.http import MediaFileUpload
    nombre = os.path.basename(ruta_local)
    mime = "application/zip" if nombre.lower().endswith(".zip") else "text/csv"
    nombre_q = nombre.replace("\\", "\\\\").replace("'", "\\'")
    ultimo_error = ""
    for intento in range(1, max_reintentos + 1):
        try:
            existentes = servicio.files().list(
                q=f"name = '{nombre_q}' and '{folder_id}' in parents and trashed = false",
                fields="files(id)", supportsAllDrives=True,
                includeItemsFromAllDrives=True).execute().get("files", [])
            media = MediaFileUpload(ruta_local, mimetype=mime, resumable=True)
            if existentes:
                servicio.files().update(fileId=existentes[0]["id"], media_body=media,
                                        fields="id", supportsAllDrives=True).execute()
                accion = "actualizado"
            else:
                servicio.files().create(body={"name": nombre, "parents": [folder_id]},
                                        media_body=media, fields="id",
                                        supportsAllDrives=True).execute()
                accion = "creado"
            drive_subidos.append(nombre)
            print(f"   ☁️ Drive ({accion}): {nombre}")
            return True
        except HttpError as e:
            ultimo_error = f"HTTP {e.resp.status}"
            if e.resp.status not in STATUS_REINTENTABLES:
                break
        except Exception as e:
            ultimo_error = str(e)
        if intento < max_reintentos:
            espera = 2 ** intento + random.random()
            print(f"   ↻ Reintento {intento}/{max_reintentos - 1} en {espera:.1f}s ({ultimo_error})")
            time.sleep(espera)
    drive_fallidos.append((nombre, ultimo_error))
    print(f"   ❌ No se pudo subir {nombre} ({ultimo_error}). Sigue en {ruta_local}")
    return False

# ── 3. NAVEGACIÓN EN CONSULTA AMIGABLE ───────────────────────────────────────
URL_BASE = "https://apps5.mineco.gob.pe/transparencia/mensual/Navegar_6.aspx"
sesion = requests.Session()
sesion.headers.update({"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                                     "(KHTML, like Gecko) Chrome/124.0 Safari/537.36",
                       "Referer": "https://apps5.mineco.gob.pe/transparencia/mensual/default.aspx"})

# Variantes de URL que se prueban una vez (calibrar) hasta que una traiga la tabla.
FORMATOS = [
    {"nombre": "navegador GET", "metodo": "GET",
     "pre": {"_tgt": "frame", "_uhc": "yes"}, "post": {"ap": "ActProy"}},
    {"nombre": "GET + ap", "metodo": "GET", "pre": {}, "post": {"ap": "ActProy"}},
    {"nombre": "GET original", "metodo": "GET", "pre": {}, "post": {}},
]
FORMATO = None                 # lo fija calibrar()

def a_numero(txt):
    t = (txt or "").replace(",", "").replace("\xa0", "").strip()
    if t in ("", "-"):
        return 0.0
    try:
        return float(t)
    except ValueError:
        return np.nan

def normalizar(txt):
    t = unicodedata.normalize("NFKD", txt or "")
    return "".join(c for c in t if not unicodedata.combining(c)).lower().strip()

_FILTRO = re.compile(r"^(total|gen[eé]rica|nivel de gobierno|gob\.?\s?loc|departamento|"
                     r"provincia|distrito|funci[oó]n|mes\b|trimestre|sector|pliego|ejecutora|"
                     r"fuente|rubro|categor|producto|divisi[oó]n|grupo funcional)", re.I)
_CODIGO = re.compile(r"^([\w\-\.]+)\s*:")

class Pagina:
    """Respuesta de Consulta Amigable ya parseada.
    filtros: breadcrumb [(etiqueta, pim)]   filas: detalle [(etiqueta, pim)]
    Columnas: … | PIA | PIM | Certificación | Comp. Anual | Atención Mensual | Devengado | Girado | Avance %
    -> el PIM es la 7.ª celda desde el final."""
    def __init__(self, html):
        s = BeautifulSoup(html, "html.parser")
        self.tiene_tabla = s.find("table", class_="Data") is not None
        self.filtros, self.filas = [], []
        for tr in s.find_all("tr"):
            if tr.find("table"):
                continue
            celdas = tr.find_all("td")
            if len(celdas) < 8:
                continue
            pim = a_numero(celdas[-7].get_text(strip=True))
            if pim != pim:                                     # NaN -> no es fila numérica
                continue
            textos = [" ".join(c.get_text(" ", strip=True).split()) for c in celdas[:3]]
            etiqueta = next((t for t in textos if t), "")
            radio = tr.find("input", attrs={"type": "radio"})
            if _FILTRO.match(etiqueta) and radio is None:
                self.filtros.append((etiqueta, pim))
            elif radio is not None or _CODIGO.match(etiqueta):
                self.filas.append((etiqueta, pim))

def consultar(anio, depto, nivel, formato=None, reintentos=4):
    """Lista de funciones (con su PIM) de un departamento; nivel=None -> todos los niveles."""
    formato = formato or FORMATO
    p = dict(formato["pre"])
    p.update({"0": "", "24": GENERICA, CLAVE_DEPTO: depto})
    if nivel:
        p["1"] = nivel
    p.update({CLAVE_FUNCION: "", "y": anio})
    p.update(formato["post"])
    p.update(PARAMS_EXTRA)
    ultimo = ""
    for i in range(reintentos):
        try:
            r = sesion.request(formato["metodo"], URL_BASE, params=p, timeout=40)
            if r.status_code == 200:
                pag = Pagina(r.content)
                if pag.tiene_tabla:
                    return pag, None
                ultimo = "SIN_TABLA (respuesta sin datos: posible bloqueo o error del MEF)"
            else:
                ultimo = f"HTTP {r.status_code}"
        except requests.RequestException as e:
            ultimo = f"{type(e).__name__}: {e}"
        time.sleep(min(30, 2 ** i + random.random()))
    return None, ultimo

def validar(p, depto, nivel):
    errores = []
    etiquetas = [normalizar(e) for e, _ in p.filtros]
    if not any(e.startswith("departamento") and f"{depto}:" in e for e in etiquetas):
        errores.append(f"la página no corresponde al departamento {depto}")
    if nivel and not any(e.startswith("nivel de gobierno") and f" {nivel.lower()}:" in e for e in etiquetas):
        errores.append(f"la página no está filtrada por el nivel {nivel}")
    return errores

def calibrar(anio, depto, nivel):
    global FORMATO
    for f in FORMATOS:
        p, err = consultar(anio, depto, nivel, formato=f, reintentos=2)
        if p and p.filas and not validar(p, depto, nivel):
            FORMATO = f
            return True
        time.sleep(random.uniform(*PAUSA))
    return False

def extraer(anio, depto, nivel):
    """-> (filas [(etiqueta, pim)], error, aviso)."""
    p, err = consultar(anio, depto, nivel)
    time.sleep(random.uniform(*PAUSA))
    if err:
        return [], err, None
    errores = validar(p, depto, nivel)
    if errores:
        return [], "VALIDACIÓN: " + "; ".join(errores), None
    aviso = None
    if p.filas and p.filtros:
        total, suma = p.filtros[-1][1], sum(v for _, v in p.filas)
        if abs(suma - total) > max(1, len(p.filas)):
            aviso = f"CUADRE: suma de funciones {suma:,.0f} vs {p.filtros[-1][0][:40]} {total:,.0f}"
    return p.filas, None, aviso

def cod_funcion(etiqueta):
    m = _CODIGO.match(etiqueta)
    return m.group(1) if m else etiqueta[:2]

# ── 4. CONEXIONES ─────────────────────────────────────────────────────────────
carpetas_drive = [f for f in (DRIVE_FOLDER_ID, DRIVE_FOLDER_CONTROL_ID) if f]
drive = conectar_drive(carpetas_drive) if SUBIR_A_DRIVE and carpetas_drive else None
SUBIR_DATOS = bool(drive and DRIVE_FOLDER_ID)
if not SUBIR_DATOS and DESCARGA_FINAL == "ninguna":
    DESCARGA_FINAL = "zip"     # sin carpeta de datos en Drive: se descargan para no perderlos

if not calibrar(años[0], departamentos[0]["codigo"], niveles[0]["codigo"]):
    raise RuntimeError("Consulta Amigable no devolvió la tabla de detalle con ningún formato de URL.")
print(f"🔗 Consulta Amigable OK (formato: {FORMATO['nombre']})\n")

# ── 5. EXTRACCIÓN: AÑO x DEPARTAMENTO x NIVEL (1 consulta = todas las funciones) ─
errores_totales, avisos_totales, controles = [], [], []
archivos_generados, resumen = [], []

for anio in años:
    regs_anio, errs_anio = [], []
    for d in tqdm(departamentos, desc=f"PIM {anio}"):
        for n in niveles:
            filas, err, aviso = extraer(anio, d["codigo"], n["codigo"])
            if err:
                errs_anio.append({"AÑO": anio, "DEPARTAMENTO": d["departamento"],
                                  "NIVEL DE GOBIERNO": n["nivel"], "ERROR": err})
            if aviso:
                avisos_totales.append({"AÑO": anio, "DEPARTAMENTO": d["departamento"],
                                       "NIVEL DE GOBIERNO": n["nivel"], "AVISO": aviso})
            for etiqueta, pim in filas:
                regs_anio.append({"AÑO": anio, "DEPARTAMENTO": d["departamento"],
                                  "NIVEL DE GOBIERNO": n["nivel"], "COD_NIVEL": n["codigo"],
                                  "FUNCION": etiqueta, "COD_FUNCION": cod_funcion(etiqueta),
                                  "PIM": pim})

        # Control: total del departamento SIN filtro de nivel vs suma de los 3 niveles
        if CONTROL:
            ref, err, _ = extraer(anio, d["codigo"], None)
            if err:
                errs_anio.append({"AÑO": anio, "DEPARTAMENTO": d["departamento"],
                                  "NIVEL DE GOBIERNO": "TODOS (control)", "ERROR": err})
            else:
                sub = [r for r in regs_anio if r["DEPARTAMENTO"] == d["departamento"]]
                ref_map = {cod_funcion(e): (e, v) for e, v in ref}
                cods = sorted(set(ref_map) | {r["COD_FUNCION"] for r in sub})
                for cod in cods:
                    fila = {"AÑO": anio, "DEPARTAMENTO": d["departamento"], "COD_FUNCION": cod,
                            "FUNCION": ref_map.get(cod, ("", 0))[0] or next(
                                (r["FUNCION"] for r in sub if r["COD_FUNCION"] == cod), "")}
                    for n in niveles:
                        fila[f"PIM_{n['nivel'].replace(' ', '_')}"] = sum(
                            r["PIM"] for r in sub if r["COD_FUNCION"] == cod and r["COD_NIVEL"] == n["codigo"])
                    fila["PIM_SUMA_NIVELES"] = sum(fila[f"PIM_{n['nivel'].replace(' ', '_')}"] for n in niveles)
                    fila["PIM_REF_TOTAL"] = ref_map.get(cod, ("", 0.0))[1]
                    fila["DIF"] = fila["PIM_SUMA_NIVELES"] - fila["PIM_REF_TOTAL"]
                    controles.append(fila)

    errores_totales.extend(errs_anio)
    df_anio = pd.DataFrame(regs_anio)
    if df_anio.empty:
        estado = "SIN DATOS"
    else:
        df_anio = df_anio.sort_values(["DEPARTAMENTO", "COD_NIVEL", "COD_FUNCION"])
        archivo = os.path.join(ruta_guardado, nombre_archivo_salida(anio))
        df_anio[COLUMNAS_SALIDA].to_csv(archivo, index=False, encoding="utf-8-sig")
        archivos_generados.append(archivo)
        if SUBIR_DATOS:
            subir_a_drive(drive, archivo, DRIVE_FOLDER_ID)
        estado = "OK" if not errs_anio else "INCOMPLETO (con errores)"
    resumen.append({"AÑO": anio, "FILAS": len(df_anio), "ERRORES": len(errs_anio), "ESTADO": estado})

print("═" * 70)
print("RESUMEN PIM")
display(pd.DataFrame(resumen))
if errores_totales:
    print("⚠️ Consultas con error:")
    with pd.option_context("display.max_colwidth", None):
        display(pd.DataFrame(errores_totales))
if avisos_totales:
    print("⚠️ Avisos de cuadre interno (suma de funciones ≠ total del filtro):")
    with pd.option_context("display.max_colwidth", None):
        display(pd.DataFrame(avisos_totales))
if not errores_totales and not avisos_totales:
    print("✅ Sin errores y con cuadre interno en todas las consultas.")

# ── 6. CONTROL + CIERRE (guardado, Drive, descarga) ──────────────────────────
TOLERANCIA = 3                 # S/: redondeo de los 3 niveles (el MEF muestra montos sin decimales)
etiqueta_años = f"{AÑO_INICIO}" if AÑO_INICIO == AÑO_FIN else f"{AÑO_INICIO}-{AÑO_FIN}"
if CONTROL and controles:
    ctl = pd.DataFrame(controles)
    ctl["ESTADO"] = np.where(ctl["DIF"].abs().le(TOLERANCIA), "OK", "REVISAR")
    ctl["TOLERANCIA_S/"] = TOLERANCIA
    ctl["FECHA_EXTRACCION"] = date.today().isoformat()
    print("\n🔍 Control: suma de niveles (E + M + R) vs total del departamento")
    display(ctl["ESTADO"].value_counts().to_frame("funciones"))
    if (ctl["ESTADO"] == "REVISAR").any():
        display(ctl[ctl["ESTADO"] == "REVISAR"])
    archivo_ctl = os.path.join(ruta_guardado, f"CONTROL PIM Funcion {etiqueta_años} - {SUFIJO_REGIONES}.csv")
    ctl.to_csv(archivo_ctl, index=False, encoding="utf-8-sig")
    archivos_generados.append(archivo_ctl)
    if drive:
        subir_a_drive(drive, archivo_ctl, DRIVE_FOLDER_CONTROL_ID)
elif CONTROL:
    print("\n⚠️ No se pudo armar el control (sin referencia del MEF).")

if errores_totales or avisos_totales:
    log = pd.concat([pd.DataFrame(errores_totales), pd.DataFrame(avisos_totales)], ignore_index=True)
    archivo_log = os.path.join(ruta_guardado, f"LOG PIM Funcion {etiqueta_años} - {SUFIJO_REGIONES}.csv")
    log.to_csv(archivo_log, index=False, encoding="utf-8-sig")
    archivos_generados.append(archivo_log)
    if SUBIR_DATOS:
        subir_a_drive(drive, archivo_log, DRIVE_FOLDER_ID)

if drive:
    print(f"\n☁️ DRIVE: {len(drive_subidos)} subido(s), {len(drive_fallidos)} fallido(s)")
    if SUBIR_DATOS:
        print(f"   Datos:   https://drive.google.com/drive/folders/{DRIVE_FOLDER_ID}")
    print(f"   Control: https://drive.google.com/drive/folders/{DRIVE_FOLDER_CONTROL_ID}")

if not archivos_generados:
    print("\n⚠️ No hay archivos para descargar.")
elif DESCARGA_FINAL == "zip":
    from google.colab import files
    ruta_zip = os.path.join(ruta_guardado, f"PIM Funcion {etiqueta_años} - {SUFIJO_REGIONES}.zip")
    with zipfile.ZipFile(ruta_zip, "w", zipfile.ZIP_DEFLATED) as z:
        for a in archivos_generados:
            z.write(a, arcname=os.path.basename(a))
    files.download(ruta_zip)
elif DESCARGA_FINAL == "individual":
    from google.colab import files
    for a in archivos_generados:
        files.download(a)
else:
    print(f"\n💾 Archivos guardados en {ruta_guardado}")